# D_S3.1 — Full-Case Classification Tree

Apply the classification tree from D_S3 to **all** cases (all SNR levels, all replicates),
and report per-family proportions at each step.

Tree: Global Relationship → Simple/Complex → Direction → Monotonicity → Line Detection → Curvature

In [1]:
# ── Imports and paths ──
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['figure.dpi'] = 120

S1_DIR = Path('output/S1')
S2_DIR = Path('output/S2')

## Load Data

In [2]:
# ── Data selection settings ──
USE_STRONG   = False
USE_STANDARD = False
USE_MILD     = True

cases = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
metrics = pd.read_parquet(S2_DIR / 'metrics_full.parquet')

meta_cols = ['case_id', 'family_id', 'family_name', 'snr', 'spread_pattern',
             'x_distribution', 'category', 'direction', 'linearity',
             'monotonicity', 'special_shape', 'replicate', 'variant_level']
meta = cases[[c for c in meta_cols if c in cases.columns]].copy()

df = metrics.merge(meta, on='case_id', how='left')
df['snr_val'] = df['snr'].apply(lambda x: float('inf') if str(x) == 'inf' else float(x))

allowed_levels = []
if USE_STRONG:   allowed_levels.append('strong')
if USE_STANDARD: allowed_levels.append('standard')
if USE_MILD:     allowed_levels.append('mild')
allowed_levels.append('none')

df = df[df['variant_level'].isin(allowed_levels)].copy()

signal = df[df['family_id'] != 'Null'].copy()
null_df = df[df['category'] == 'true_null'].copy()

SHORT = {
    'F01': 'Linear(+)',   'F02': 'Linear(-)',   'F03': 'PwrCvx(+)',
    'F04': 'PwrCvx(-)',   'F05': 'PwrCcv(+)',   'F06': 'PwrCcv(-)',
    'F07': 'Sat(+)',      'F08': 'Sat(-)',       'F09': 'Log(+)',
    'F10': 'Log(-)',      'F11': 'Exp(+)',       'F12': 'Exp(-)',
    'F13': 'S-crv(+)',    'F14': 'S-crv(-)',     'F15': 'Thresh(+)',
    'F16': 'Thresh(-)',   'F17': 'Peak',         'F18': 'Valley',
    'F19': 'Spike',       'F20': 'Inv Spike',    'F21': 'Cubic',
    'F22': 'Oscillation', 'F23': 'Two Lines',    'F24': 'L+Parab',
    'F25': 'Multi-reg',   'F26': 'Windowed J',   'Null': 'No relationship',
}

print(f'Variant levels: {allowed_levels}')
print(f'Total: {len(df):,} cases')
print(f'Signal (26 families): {len(signal):,}')
print(f'Null: {len(null_df):,}')

Variant levels: ['mild', 'none']
Total: 54,270 cases
Signal (26 families): 52,260
Null: 2,010


In [3]:
# ── Unified color palette ──
C_GLOBAL     = '#1976D2'
C_SIMPLE     = '#2E7D32'
C_COMPLEX    = '#EF6C00'
C_NO_REL     = '#BDBDBD'
C_NULL       = '#7E7E7E'
C_POS        = '#4FC3F7'
C_NEG        = '#EF5350'
C_MONO_S     = '#66BB6A'
C_MONO_M     = '#FFA726'
C_LINE       = '#26C6DA'
C_NONLINE    = '#6A1B9A'
C_CONVEX     = '#2E7D32'
C_CONCAVE    = '#00897B'
C_INFLECTED  = '#EF6C00'
C_WEAK       = '#757575'

In [4]:
# ── Helper: two-column family table ──
def print_family_table(data_df, fids, columns, header_fmt, row_fmt, majority_fn=None):
    """Print a two-column table of per-family counts."""
    hdr = header_fmt
    sep = '─' * len(hdr)
    rows = []
    for fid in fids:
        sub = data_df[data_df['family_id'] == fid]
        rows.append(row_fmt(fid, sub))

    mid = (len(rows) + 1) // 2
    col_w = len(hdr)
    print(f'{hdr:<{col_w}}  {hdr}')
    print(f'{sep}  {sep}')
    for i in range(mid):
        left = rows[i]
        right = rows[i + mid] if i + mid < len(rows) else ''
        print(f'{left:<{col_w}}  {right}')
    print()

def fmt_count_pct(count, total, width=5):
    """Return 'count(pct)' with a safe denominator."""
    if total == 0:
        return f'{count:>{width},}(  —)'
    return f'{count:>{width},}({count/total:>4.0%})'



## Step 1 — Global Relationship

`MIC >= 0.8 AND distance_correlation >= 0`

In [5]:
# ── Step 1: Global Relationship ──
MIC_THRESH = 0.8
DCOR_THRESH = 0

mic_pass  = signal['MIC'] >= MIC_THRESH
dcor_pass = signal['distance_correlation'] >= DCOR_THRESH
both_pass = mic_pass & dcor_pass

filtered_cases = signal[both_pass].copy()

all_fids = sorted(signal['family_id'].unique())
qual_fids = sorted(filtered_cases['family_id'].unique())
excl_fids = [f for f in all_fids if f not in qual_fids]

print(f'═══ Step 1: Global Relationship ═══')
print(f'Rule: MIC ≥ {MIC_THRESH} AND dcor ≥ {DCOR_THRESH}  (per case)')
print()
print(f'  MIC ≥ {MIC_THRESH}:  {mic_pass.sum():>7,} / {len(signal):,}  ({mic_pass.mean():.1%})')
print(f'  dcor ≥ {DCOR_THRESH}: {dcor_pass.sum():>7,} / {len(signal):,}  ({dcor_pass.mean():.1%})')
print(f'  Both:     {both_pass.sum():>7,} / {len(signal):,}  ({both_pass.mean():.1%})')
print(f'  Qualifying families: {len(qual_fids)} / {len(all_fids)}  ({len(qual_fids)/len(all_fids):.1%}) — {qual_fids}')
print(f'  Excluded: {excl_fids}')
print()

hdr = f'{"Family":<16} {"MIC≥.8":>12} {"dcor≥0":>12} {"Both":>12} {"Total":>7}'
def _row_s1(fid, sub):
    n = len(sub)
    n_mic  = int((sub['MIC'] >= MIC_THRESH).sum())
    n_dcor = int((sub['distance_correlation'] >= DCOR_THRESH).sum())
    n_both = int(((sub['MIC'] >= MIC_THRESH) & (sub['distance_correlation'] >= DCOR_THRESH)).sum())
    return f'{fid} {SHORT[fid]:<10} {n_mic:>5,}({n_mic/n:>4.0%}) {n_dcor:>5,}({n_dcor/n:>4.0%}) {n_both:>5,}({n_both/n:>4.0%}) {n:>7,}'

print_family_table(signal, all_fids, None, hdr, _row_s1)


═══ Step 1: Global Relationship ═══
Rule: MIC ≥ 0.8 AND dcor ≥ 0  (per case)

  MIC ≥ 0.8:   15,419 / 52,260  (29.5%)
  dcor ≥ 0:  52,260 / 52,260  (100.0%)
  Both:      15,419 / 52,260  (29.5%)
  Qualifying families: 25 / 26  (96.2%) — ['F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F20', 'F21', 'F22', 'F23', 'F25', 'F26']
  Excluded: ['F24']

Family                 MIC≥.8       dcor≥0         Both   Total  Family                 MIC≥.8       dcor≥0         Both   Total
───────────────────────────────────────────────────────────────  ───────────────────────────────────────────────────────────────
F01 Linear(+)    716( 36%) 2,010(100%)   716( 36%)   2,010       F14 S-crv(-)     775( 39%) 2,010(100%)   775( 39%)   2,010
F02 Linear(-)    720( 36%) 2,010(100%)   720( 36%)   2,010       F15 Thresh(+)    817( 41%) 2,010(100%)   817( 41%)   2,010
F03 PwrCvx(+)    715( 36%) 2,010(100%)   715( 36%)   2,010  

## Step 2 — Simple vs Complex

Base: `|ρ| ≥ 0.7 AND |r| ≥ 0.7`
Optional gates (toggleable):
- `MAS < 0.10` — non-monotonicity filter
- `MIC-r² < 0.20` — non-linearity filter (catches threshold-type)

In [6]:
# ── Step 2: Simple vs Complex ──
R_THRESH = 0.7
REVERSAL_PROMINENCE = 0.10

USE_MAS_GATE = False
MAS_THRESH = 0.20

USE_MR_GATE = False
MR_THRESH = 0.20

sc = filtered_cases.copy()

cond_rho = sc['spearman_rho'].abs() >= R_THRESH
cond_r   = sc['pearson_r'].abs() >= R_THRESH
cond_mas = sc['MAS'] < MAS_THRESH if USE_MAS_GATE else True
cond_mr  = sc['MIC_minus_r2'] < MR_THRESH if USE_MR_GATE else True

E_s = sc['standardized_polyfit_early_slope'].values
M_s = sc['standardized_polyfit_middle_slope'].values
L_s = sc['standardized_polyfit_late_slope'].values

sign_E, sign_M, sign_L = np.sign(E_s), np.sign(M_s), np.sign(L_s)
has_sign_change = ~((sign_E == sign_M) & (sign_M == sign_L))

abs_max = np.maximum(np.maximum(np.abs(E_s), np.abs(M_s)), np.abs(L_s))
abs_max = np.where(abs_max == 0, 1, abs_max)
maj_sign = np.sign(E_s + M_s + L_s)

rev_prom = np.zeros(len(sc))
for i in range(len(sc)):
    if not has_sign_change[i] or maj_sign[i] == 0:
        continue
    rev_vals = [abs(s) for s in [E_s[i], M_s[i], L_s[i]]
                if np.sign(s) != maj_sign[i] and np.sign(s) != 0]
    if rev_vals:
        rev_prom[i] = max(rev_vals) / abs_max[i]

sc['reversal_prominence'] = rev_prom
cond_no_turn = pd.Series(~(has_sign_change & (rev_prom >= REVERSAL_PROMINENCE)), index=sc.index)

sc['is_simple'] = cond_rho & cond_r & cond_mas & cond_mr & cond_no_turn
simple_cases = sc[sc['is_simple']].copy()

simple_fids = sorted(simple_cases['family_id'].unique())
complex_fids = [f for f in qual_fids if f not in simple_fids]

rule_parts = [f'|ρ| ≥ {R_THRESH}', f'|r| ≥ {R_THRESH}']
if USE_MAS_GATE: rule_parts.append(f'MAS < {MAS_THRESH}')
if USE_MR_GATE:  rule_parts.append(f'MIC-r² < {MR_THRESH}')
rule_str = ' AND '.join(rule_parts)

print(f'═══ Step 2: Simple vs Complex ═══')
print(f'Rule: {rule_str}')
print(f'      + no slope sign reversal (prominence ≥ {REVERSAL_PROMINENCE})')
print(f'Gates: MAS={USE_MAS_GATE}, MIC-r²={USE_MR_GATE}')
print()
print(f'  Simple: {len(simple_cases):>7,} / {len(filtered_cases):,}  ({len(simple_cases)/len(filtered_cases):.1%})')
print(f'  Complex: {len(filtered_cases)-len(simple_cases):>7,} / {len(filtered_cases):,}')
print(f'  Simple families: {len(simple_fids)} — {simple_fids}')
print(f'  Complex-only families: {complex_fids}')
print()

# Per-family breakdown with each metric as a column (two-column layout)
def _build_s2_row(fid, sub):
    n = len(sub)
    if n == 0:
        return None
    n_rho = int((sub['spearman_rho'].abs() >= R_THRESH).sum())
    n_r   = int((sub['pearson_r'].abs() >= R_THRESH).sum())
    vals = [f'{n_rho:>5}({n_rho/n:>4.0%})', f'{n_r:>5}({n_r/n:>4.0%})']
    if USE_MAS_GATE:
        n_mas = int((sub['MAS'] < MAS_THRESH).sum())
        vals.append(f'{n_mas:>5}({n_mas/n:>4.0%})')
    if USE_MR_GATE:
        n_mr = int((sub['MIC_minus_r2'] < MR_THRESH).sum())
        vals.append(f'{n_mr:>5}({n_mr/n:>4.0%})')
    has_sc = ~((np.sign(sub['standardized_polyfit_early_slope'].values) ==
                np.sign(sub['standardized_polyfit_middle_slope'].values)) &
               (np.sign(sub['standardized_polyfit_middle_slope'].values) ==
                np.sign(sub['standardized_polyfit_late_slope'].values)))
    n_norev = int((~(has_sc & (sub['reversal_prominence'].values >= REVERSAL_PROMINENCE))).sum())
    vals.append(f'{n_norev:>5}({n_norev/n:>4.0%})')
    n_all = int(sub['is_simple'].sum())
    vals.append(f'{n_all:>5}({n_all/n:>4.0%})')
    vals.append(f'{n:>7,}')
    return f'{fid} {SHORT[fid]:<10}' + ''.join(f'{v:>11}' for v in vals)

col_labels = [f'|ρ|≥{R_THRESH}', f'|r|≥{R_THRESH}']
if USE_MAS_GATE: col_labels.append(f'MAS<{MAS_THRESH}')
if USE_MR_GATE:  col_labels.append(f'MR<{MR_THRESH}')
col_labels += ['NoRev', 'ALL', 'Total']

hdr = f'{"Family":<16}' + ''.join(f'{c:>11}' for c in col_labels)
sep = '─' * len(hdr)

rows = []
for fid in qual_fids:
    sub = sc[sc['family_id'] == fid]
    row = _build_s2_row(fid, sub)
    if row:
        rows.append(row)

mid = (len(rows) + 1) // 2
col_w = len(hdr)
print(f'{hdr:<{col_w}}  {hdr}')
print(f'{sep}  {sep}')
for i in range(mid):
    left = rows[i]
    right = rows[i + mid] if i + mid < len(rows) else ''
    print(f'{left:<{col_w}}  {right}')

═══ Step 2: Simple vs Complex ═══
Rule: |ρ| ≥ 0.7 AND |r| ≥ 0.7
      + no slope sign reversal (prominence ≥ 0.1)
Gates: MAS=False, MIC-r²=False

  Simple:  11,527 / 15,419  (74.8%)
  Complex:   3,892 / 15,419
  Simple families: 18 — ['F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F21', 'F23']
  Complex-only families: ['F17', 'F18', 'F19', 'F20', 'F22', 'F25', 'F26']

Family              |ρ|≥0.7    |r|≥0.7      NoRev        ALL      Total  Family              |ρ|≥0.7    |r|≥0.7      NoRev        ALL      Total
───────────────────────────────────────────────────────────────────────  ───────────────────────────────────────────────────────────────────────
F01 Linear(+)   716(100%)  716(100%)  716(100%)  716(100%)        716    F14 S-crv(-)    775(100%)  775(100%)  775(100%)  775(100%)        775
F02 Linear(-)   720(100%)  720(100%)  720(100%)  720(100%)        720    F15 Thresh(+)   817(100%)  817(100%)  798( 98%)  798( 98%

## Step 3 — Direction

Sign of Pearson r. Only applies to Simple cases.

In [7]:
# ── Step 3: Direction ──
simple_cases['direction_tree'] = np.where(simple_cases['pearson_r'] > 0, 'Positive', 'Negative')

print(f'═══ Step 3: Direction (sign of Pearson r) ═══')
print()

hdr = f'{"Family":<16} {"Pos(+)":>12} {"Neg(−)":>12} {"Total":>7} {"Majority":>9}'
def _row_s3(fid, sub):
    n = len(sub)
    n_pos = int((sub['direction_tree'] == 'Positive').sum())
    n_neg = int((sub['direction_tree'] == 'Negative').sum())
    maj = '+' if n_pos >= n_neg else '−'
    return f'{fid} {SHORT[fid]:<10} {fmt_count_pct(n_pos,n):>12} {fmt_count_pct(n_neg,n):>12} {n:>7,} {maj:>9}'

print_family_table(simple_cases, simple_fids, None, hdr, _row_s3)


═══ Step 3: Direction (sign of Pearson r) ═══

Family                 Pos(+)       Neg(−)   Total  Majority  Family                 Pos(+)       Neg(−)   Total  Majority
────────────────────────────────────────────────────────────  ────────────────────────────────────────────────────────────
F01 Linear(+)     716(100%)      0(  0%)     716         +    F10 Log(-)          0(  0%)    675(100%)     675         −
F02 Linear(-)       0(  0%)    720(100%)     720         −    F11 Exp(+)        698(100%)      0(  0%)     698         +
F03 PwrCvx(+)     715(100%)      0(  0%)     715         +    F12 Exp(-)          0(  0%)    701(100%)     701         −
F04 PwrCvx(-)       0(  0%)    721(100%)     721         −    F13 S-crv(+)      767(100%)      0(  0%)     767         +
F05 PwrCcv(+)     700(100%)      0(  0%)     700         +    F14 S-crv(-)        0(  0%)    775(100%)     775         −
F06 PwrCcv(-)       0(  0%)    698(100%)     698         −    F15 Thresh(+)     798(100%)      0(  0%)

## Step 4 — Monotonicity

|Spearman ρ| level: **Strong** (≥ 0.9) vs **Med** (< 0.9).

In [8]:
# ── Step 4: Monotonicity ──
MONO_STRONG = 0.9

simple_cases['monotonicity_tree'] = np.where(
    simple_cases['spearman_rho'].abs() >= MONO_STRONG, 'Strong', 'Med'
)

print(f'═══ Step 4: Monotonicity (|ρ| ≥ {MONO_STRONG} → Strong) ═══')
print()

hdr = f'{"Family":<16} {"Strong":>12} {"Med":>12} {"Total":>7} {"Majority":>9}'
def _row_s4(fid, sub):
    n = len(sub)
    n_s = int((sub['monotonicity_tree'] == 'Strong').sum())
    n_m = int((sub['monotonicity_tree'] == 'Med').sum())
    maj = 'Strong' if n_s >= n_m else 'Med'
    return f'{fid} {SHORT[fid]:<10} {fmt_count_pct(n_s,n):>12} {fmt_count_pct(n_m,n):>12} {n:>7,} {maj:>9}'

print_family_table(simple_cases, simple_fids, None, hdr, _row_s4)


═══ Step 4: Monotonicity (|ρ| ≥ 0.9 → Strong) ═══

Family                 Strong          Med   Total  Majority  Family                 Strong          Med   Total  Majority
────────────────────────────────────────────────────────────  ────────────────────────────────────────────────────────────
F01 Linear(+)     715(100%)      1(  0%)     716    Strong    F10 Log(-)        675(100%)      0(  0%)     675    Strong
F02 Linear(-)     716( 99%)      4(  1%)     720    Strong    F11 Exp(+)        696(100%)      2(  0%)     698    Strong
F03 PwrCvx(+)     711( 99%)      4(  1%)     715    Strong    F12 Exp(-)        697( 99%)      4(  1%)     701    Strong
F04 PwrCvx(-)     714( 99%)      7(  1%)     721    Strong    F13 S-crv(+)      726( 95%)     41(  5%)     767    Strong
F05 PwrCcv(+)     698(100%)      2(  0%)     700    Strong    F14 S-crv(-)      729( 94%)     46(  6%)     775    Strong
F06 PwrCcv(-)     698(100%)      0(  0%)     698    Strong    F15 Thresh(+)     259( 32%)    539( 

## Step 5 — Line Detection

**Line** = Strong monotonicity + |r| ≥ 0.9 + slope_span < 0.20 + MIC-r² < 0.08

MIC-r² confirms linearity: low means r² already explains what MIC captures.

In [9]:
# ── Step 5: Line Detection ──
LIN_STRONG = 0.9
LINE_SPAN_TOL = 0.20
LINE_MR_THRESH = 1
simple_cases['linearity_tree'] = np.where(
    simple_cases['pearson_r'].abs() >= LIN_STRONG, 'High |r|', 'Med |r|'
)

E_raw = simple_cases['standardized_polyfit_early_slope'].values
M_raw = simple_cases['standardized_polyfit_middle_slope'].values
L_raw = simple_cases['standardized_polyfit_late_slope'].values

scale = np.maximum(np.maximum(np.abs(E_raw), np.abs(M_raw)), np.abs(L_raw))
scale = np.where(scale == 0, 1, scale)
e_n, m_n, l_n = E_raw / scale, M_raw / scale, L_raw / scale

dsign = np.where(simple_cases['pearson_r'].values > 0, 1, -1)
e_d = e_n * dsign
m_d = m_n * dsign
l_d = l_n * dsign

simple_cases['e_d'] = e_d
simple_cases['m_d'] = m_d
simple_cases['l_d'] = l_d

slopes = np.column_stack([e_d, m_d, l_d])
slope_span = slopes.max(axis=1) - slopes.min(axis=1)
simple_cases['slope_span'] = slope_span

simple_cases['is_line'] = (
    (simple_cases['monotonicity_tree'] == 'Strong') &
    (simple_cases['linearity_tree'] == 'High |r|') &
    (simple_cases['slope_span'] < LINE_SPAN_TOL) &
    (simple_cases['MIC_minus_r2'] < LINE_MR_THRESH)
)

print(f'═══ Step 5: Line Detection ═══')
print(f'Rule: Strong mono + |r| ≥ {LIN_STRONG} + slope_span < {LINE_SPAN_TOL} + MIC-r² < {LINE_MR_THRESH}')
print(f'Line: {simple_cases["is_line"].sum():,} / {len(simple_cases):,}  ({simple_cases["is_line"].mean():.1%})')
print()

hdr = f'{"Family":<16} {"Line":>7} {"Non-L":>7} {"Total":>7} {"Majority":>9}'
def _row_s5(fid, sub):
    n = len(sub)
    n_l = int(sub['is_line'].sum())
    n_nl = n - n_l
    maj = 'Line' if n_l >= n_nl else 'Non-L'
    return f'{fid} {SHORT[fid]:<10} {n_l:>7,} {n_nl:>7,} {n:>7,} {maj:>9}'

print_family_table(simple_cases, simple_fids, None, hdr, _row_s5)

═══ Step 5: Line Detection ═══
Rule: Strong mono + |r| ≥ 0.9 + slope_span < 0.2 + MIC-r² < 1
Line: 1,390 / 11,527  (12.1%)

Family              Line   Non-L   Total  Majority  Family              Line   Non-L   Total  Majority
──────────────────────────────────────────────────  ──────────────────────────────────────────────────
F01 Linear(+)      683      33     716      Line    F10 Log(-)           0     675     675     Non-L
F02 Linear(-)      706      14     720      Line    F11 Exp(+)           0     698     698     Non-L
F03 PwrCvx(+)        0     715     715     Non-L    F12 Exp(-)           0     701     701     Non-L
F04 PwrCvx(-)        0     721     721     Non-L    F13 S-crv(+)         0     767     767     Non-L
F05 PwrCcv(+)        1     699     700     Non-L    F14 S-crv(-)         0     775     775     Non-L
F06 PwrCcv(-)        0     698     698     Non-L    F15 Thresh(+)        0     798     798     Non-L
F07 Sat(+)           0     624     624     Non-L    F16 Thresh(-

## Step 6 — Curvature

For non-Line Simple cases. Classification by E/M/L slope ordering pattern:
- **Concave**: e_d > m_d > l_d (slope decreasing — steep early, flat late)
- **Convex**: e_d < m_d < l_d (slope increasing — flat early, steep late)
- **Inflected**: m_d > e_d AND m_d > l_d (middle steepest)
- **Weak/Uncertain**: V_s too low (slopes not different enough)

In [10]:
# ── Step 6: Curvature ──
CURVE_VS_MIN = 0.35

EPS = 1e-8
slopes = np.column_stack([simple_cases['e_d'].values,
                          simple_cases['m_d'].values,
                          simple_cases['l_d'].values])
V_s = slopes.std(axis=1, ddof=0) / (np.abs(slopes.mean(axis=1)) + EPS)
simple_cases['slope_Vs'] = V_s

e_d = simple_cases['e_d'].values
m_d = simple_cases['m_d'].values
l_d = simple_cases['l_d'].values

shape = np.full(len(simple_cases), 'Weak/Uncertain', dtype=object)
shape[simple_cases['is_line'].values] = 'Line'

curve_ready = (shape == 'Weak/Uncertain') & (V_s >= CURVE_VS_MIN)

shape[curve_ready & (e_d > m_d) & (m_d > l_d)] = 'Concave'
shape[curve_ready & (e_d < m_d) & (m_d < l_d)] = 'Convex'
shape[curve_ready & (m_d > e_d) & (m_d > l_d)] = 'Inflected'

simple_cases['shape_tree'] = shape

all_shapes = ['Line', 'Convex', 'Concave', 'Inflected', 'Weak/Uncertain']

print(f'═══ Step 6: Curvature ═══')
print(f'Concave:   e_d > m_d > l_d (slope decreasing)')
print(f'Convex:    e_d < m_d < l_d (slope increasing)')
print(f'Inflected: m_d > e_d AND m_d > l_d (middle steepest)')
print(f'Gate: V_s ≥ {CURVE_VS_MIN}')
print()
for s in all_shapes:
    n = int((simple_cases['shape_tree'] == s).sum())
    print(f'  {s:<16} {n:>6,} ({n/len(simple_cases):>5.1%})')
print(f'  {"Total":<16} {len(simple_cases):>6,}')
print()

hdr = f'{"Family":<16} {"Line":>6} {"Convex":>7} {"Concave":>8} {"Inflect":>8} {"Weak":>6} {"Total":>7} {"Majority":>16}'
def _row_s6(fid, sub):
    n = len(sub)
    counts = {s: int((sub['shape_tree'] == s).sum()) for s in all_shapes}
    maj = sub['shape_tree'].mode().iloc[0] if n > 0 else '—'
    return (f'{fid} {SHORT[fid]:<10} {counts["Line"]:>6,} {counts["Convex"]:>7,} '
            f'{counts["Concave"]:>8,} {counts["Inflected"]:>8,} '
            f'{counts["Weak/Uncertain"]:>6,} {n:>7,} {maj:>16}')

print_family_table(simple_cases, simple_fids, None, hdr, _row_s6)

═══ Step 6: Curvature ═══
Concave:   e_d > m_d > l_d (slope decreasing)
Convex:    e_d < m_d < l_d (slope increasing)
Inflected: m_d > e_d AND m_d > l_d (middle steepest)
Gate: V_s ≥ 0.35

  Line              1,390 (12.1%)
  Convex            1,344 (11.7%)
  Concave           2,600 (22.6%)
  Inflected         3,051 (26.5%)
  Weak/Uncertain    3,142 (27.3%)
  Total            11,527

Family             Line  Convex  Concave  Inflect   Weak   Total         Majority  Family             Line  Convex  Concave  Inflect   Weak   Total         Majority
─────────────────────────────────────────────────────────────────────────────────  ─────────────────────────────────────────────────────────────────────────────────
F01 Linear(+)     683       0        0        0     33     716             Line    F10 Log(-)          0       0      672        0      3     675          Concave
F02 Linear(-)     706       0        0        0     14     720             Line    F11 Exp(+)          0     563        0

In [11]:
# # ── Step 6 diagnostics: visualize Weak/Uncertain curvature cases ──
# weak_cases = simple_cases[simple_cases['shape_tree'] == 'Weak/Uncertain'].copy()
# weak_cases = weak_cases.sort_values(['family_id', 'snr_val', 'replicate', 'case_id'], ascending=[True, False, True, True])

# print(f'Weak/Uncertain cases: {len(weak_cases):,}')
# if len(weak_cases) > 0:
#     weak_pts = np.load(S1_DIR / 'scatter_points.npz')
#     weak_x_all, weak_y_all = weak_pts['x'], weak_pts['y']
#     weak_case_idx = pd.Series(np.arange(len(cases)), index=cases['case_id']).to_dict()

#     ncols = min(6, len(weak_cases))
#     nrows = math.ceil(len(weak_cases) / ncols)
#     fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 2.75 * nrows), squeeze=False)
#     axes_flat = axes.ravel()

#     for ax, (_, row) in zip(axes_flat, weak_cases.iterrows()):
#         idx = weak_case_idx.get(row['case_id'])
#         if idx is None:
#             ax.set_visible(False)
#             continue

#         ax.scatter(weak_x_all[idx], weak_y_all[idx], s=6, alpha=0.45,
#                    color=C_WEAK, edgecolors='none')
#         snr_label = '∞' if row['snr_val'] == float('inf') else f'{row["snr_val"]:g}'
#         title_top = f'{row["family_id"]} {SHORT.get(row["family_id"], "")} | SNR={snr_label} | rep={int(row["replicate"])}'
#         title_bottom = f'r={row["pearson_r"]:.3f}, ρ={row["spearman_rho"]:.3f}, MIC={row["MIC"]:.3f}, Vs={row["slope_Vs"]:.2f}'
#         ax.set_title(title_top + chr(10) + title_bottom, fontsize=8.5, fontweight='bold')
#         ax.set_xticks([])
#         ax.set_yticks([])

#     for ax in axes_flat[len(weak_cases):]:
#         ax.set_visible(False)

#     fig.suptitle('Weak/Uncertain curvature cases', fontsize=14, fontweight='bold', y=1.002)
#     plt.tight_layout(h_pad=0.9, w_pad=0.45)
#     plt.show()

Weak/Uncertain cases: 3,142


ValueError: Image size of 2160x172920 pixels is too large. It must be less than 2^16 in each direction.

Error in callback <function _draw_all_if_interactive at 0x1286ac0d0> (for post_execute), with arguments args (),kwargs {}:


ValueError: Image size of 2160x172920 pixels is too large. It must be less than 2^16 in each direction.

ValueError: Image size of 2160x172920 pixels is too large. It must be less than 2^16 in each direction.

<Figure size 2160x172920 with 3144 Axes>

## Classification Summary

Per-family tree profile across all cases.

In [142]:
# ── Classification Summary ──
print(f'═══ Classification Summary ═══')
print(f'Variant levels: {[l for l in ["strong","standard","mild"] if l in filtered_cases["variant_level"].values]}')
print(f'Filter: MIC ≥ {MIC_THRESH}, dcor ≥ {DCOR_THRESH}')
print(f'Filtered: {len(filtered_cases):,} / {len(signal):,}  ({len(filtered_cases)/len(signal):.1%})')
print(f'Simple:   {len(simple_cases):,} / {len(filtered_cases):,}  ({len(simple_cases)/len(filtered_cases):.1%})')
print()

hdr = f'{"Family":<16} {"Filter":>12} {"Simple":>12} {"Dir":>4} {"Mono":>6} {"Shape":>16} {"Profile"}'
print(hdr)
print('─' * len(hdr))
for fid in qual_fids:
    filt_n = len(filtered_cases[filtered_cases['family_id'] == fid])
    fid_simple = simple_cases[simple_cases['family_id'] == fid]
    n_s = len(fid_simple)
    if n_s == 0:
        print(f'{fid} {SHORT[fid]:<10} {fmt_count_pct(filt_n, len(signal[signal["family_id"] == fid])):>12} {fmt_count_pct(n_s, filt_n):>12} {"—":>4} {"—":>6} {"—":>16}  —')
        continue
    dr = '+' if (fid_simple['direction_tree'] == 'Positive').sum() >= (fid_simple['direction_tree'] == 'Negative').sum() else '−'
    mo = fid_simple['monotonicity_tree'].mode().iloc[0]
    sh = fid_simple['shape_tree'].mode().iloc[0]
    profile = f'{dr}, {mo}, {sh}'
    print(f'{fid} {SHORT[fid]:<10} {fmt_count_pct(filt_n, len(signal[signal["family_id"] == fid])):>12} {fmt_count_pct(n_s, filt_n):>12} {dr:>4} {mo:>6} {sh:>16}  {profile}')


═══ Classification Summary ═══
Variant levels: ['strong']
Filter: MIC ≥ 0.8, dcor ≥ 0.0
Filtered: 11,859 / 52,260  (22.7%)
Simple:   9,417 / 11,859  (79.4%)

Family                 Filter       Simple  Dir   Mono            Shape Profile
───────────────────────────────────────────────────────────────────────────────
F01 Linear(+)     717( 36%)    717(100%)    + Strong             Line  +, Strong, Line
F02 Linear(-)     717( 36%)    717(100%)    − Strong             Line  −, Strong, Line
F03 PwrCvx(+)     530( 26%)    530(100%)    + Strong           Convex  +, Strong, Convex
F04 PwrCvx(-)     527( 26%)    527(100%)    − Strong           Convex  −, Strong, Convex
F05 PwrCcv(+)     601( 30%)    601(100%)    + Strong          Concave  +, Strong, Concave
F06 PwrCcv(-)     605( 30%)    605(100%)    − Strong          Concave  −, Strong, Concave
F07 Sat(+)         13(  1%)      0(  0%)    —      —                —  —
F08 Sat(-)         15(  1%)      0(  0%)    —      —                —  —
F09 

## SNR Breakdown — Shape Classification Rate by Noise Level

How does each family's dominant shape hold up as SNR decreases?

In [143]:
# ── SNR Breakdown: shape classification by SNR ──
simple_cases['snr_val'] = simple_cases['snr'].apply(
    lambda x: float('inf') if str(x) == 'inf' else float(x))

snr_levels = sorted(simple_cases['snr_val'].unique(), reverse=True)
snr_labels = [f'{s:.0f}' if np.isfinite(s) else '∞' for s in snr_levels]

all_shapes = ['Line', 'Convex', 'Concave', 'Inflected', 'Weak/Uncertain']

print(f'═══ Shape Proportion by SNR ═══')
print()

hdr_parts = [f'{"Family":<16}'] + [f'{sl:>8}' for sl in snr_labels]
hdr = ''.join(hdr_parts)
print(hdr)
print('─' * len(hdr))

for fid in simple_fids:
    fid_data = simple_cases[simple_cases['family_id'] == fid]
    parts = [f'{fid} {SHORT[fid]:<10}']
    for sv in snr_levels:
        sub = fid_data[fid_data['snr_val'] == sv]
        n = len(sub)
        if n == 0:
            parts.append(f'{"—":>8}')
        else:
            maj = sub['shape_tree'].mode().iloc[0]
            n_maj = int((sub['shape_tree'] == maj).sum())
            parts.append(f'{maj[:4]:>4}{n_maj/n:>4.0%}')
    print(''.join(parts))

═══ Shape Proportion by SNR ═══

Family                 ∞     500     468     438     410     384     360     337     315     295     276     259     242     227     212     199     186     174     163     153     143     134     125     117     110     103      96      90      84      79      74      69      65      61      57      53      50      47      44      41      38      36      33      31      29      27      26      24      23      21      20      18      17      16      15      14      13      12      12      11      10      10       9       8       8       7       7       6       6       6       5       5       5       4       4       4       4       3       3       3       3       3       2       2       2       2       2       2       2
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────